# 01 — Data Exploration

Explore the seed data loaded into BigQuery for the Cash Agent Demo.

In [ ]:
import os
from google.cloud import bigquery

PROJECT_ID = os.environ.get('PROJECT_ID', 'your-project-id')
DATASET_ID = 'cash_agent_demo'
client = bigquery.Client(project=PROJECT_ID)

## Bank Account Balances

In [ ]:
query = f"""
SELECT bank_name, account_type, currency, current_balance
FROM `{PROJECT_ID}.{DATASET_ID}.bank_accounts`
ORDER BY currency, bank_name
"""
client.query(query).to_dataframe()

## Cash Position by Currency

In [ ]:
query = f"""
SELECT currency, SUM(current_balance) AS total_balance
FROM `{PROJECT_ID}.{DATASET_ID}.bank_accounts`
GROUP BY currency
ORDER BY currency
"""
client.query(query).to_dataframe()

## AP/AR Summary

In [ ]:
# AP by currency
query = f"""
SELECT currency, COUNT(*) AS items, SUM(amount) AS total_amount
FROM `{PROJECT_ID}.{DATASET_ID}.ap_open_items`
WHERE status = 'OPEN'
GROUP BY currency
"""
print("AP Open Items:")
display(client.query(query).to_dataframe())

# AR by currency
query = f"""
SELECT currency, COUNT(*) AS items, SUM(amount) AS total_amount,
       SUM(amount * probability) AS expected_amount
FROM `{PROJECT_ID}.{DATASET_ID}.ar_open_items`
WHERE status = 'OPEN'
GROUP BY currency
"""
print("AR Open Items:")
display(client.query(query).to_dataframe())

## Risky Receivables

In [ ]:
query = f"""
SELECT customer_name, amount, currency, due_date, probability
FROM `{PROJECT_ID}.{DATASET_ID}.ar_open_items`
WHERE probability < 0.7
ORDER BY amount DESC
"""
client.query(query).to_dataframe()

## Cash Journal — Daily Net Flow (Last 30 Days)

In [ ]:
query = f"""
SELECT posting_date, currency,
       SUM(CASE WHEN transaction_type='INFLOW' THEN amount ELSE -amount END) AS net_flow
FROM `{PROJECT_ID}.{DATASET_ID}.cash_journal`
WHERE posting_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
GROUP BY posting_date, currency
ORDER BY posting_date, currency
"""
df = client.query(query).to_dataframe()
df.pivot(index='posting_date', columns='currency', values='net_flow').plot(
    figsize=(14, 5), title='Daily Net Cash Flow by Currency'
)

## FX Rates (Last 30 Days)

In [ ]:
query = f"""
SELECT rate_date, from_currency || '/' || to_currency AS pair, exchange_rate
FROM `{PROJECT_ID}.{DATASET_ID}.fx_rates`
WHERE rate_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
ORDER BY rate_date, pair
"""
df = client.query(query).to_dataframe()
df.pivot(index='rate_date', columns='pair', values='exchange_rate').plot(
    figsize=(14, 5), title='FX Rates (30 Days)', subplots=True
)